# Fashion-MNIST Classification: CNN vs CNN with Batch Normalization

**Course:** IBM AI Engineering Professional Certificate
**Module:** Deep Neural Networks with PyTorch
**Author:** Jack Pumpuni Frimpong-Manso
**Dataset:** [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)
**Estimated Time:** 30 minutes

---

## Learning Objectives

After completing this lab you will be able to:
- Build and train a Convolutional Neural Network (CNN) for multi-class image classification
- Implement Batch Normalization and explain its effect on training stability and convergence
- Compare training dynamics (cost, accuracy) of CNN vs CNN_batch across 5 epochs
- Evaluate model performance per class and identify common misclassification patterns

---

## Table of Contents

1. [Environment Setup & Dataset](#1-environment-setup--dataset)
2. [Model Architectures](#2-model-architectures)
3. [Training Both Models](#3-training-both-models)
4. [Results & Comparison](#4-results--comparison)
5. [Error Analysis](#5-error-analysis)
6. [Summary](#6-summary)

In [ ]:
%%time
# Install PyTorch (CPU-only build) and supporting libraries
# Using pinned versions to ensure reproducibility across environments
%pip install pandas numpy matplotlib --quiet
%pip install torch==2.8.0+cpu torchvision==0.23.0+cpu torchaudio==2.8.0+cpu \
    --index-url https://download.pytorch.org/whl/cpu --quiet

In [ ]:
# ── Standard library and framework imports ───────────────────────────────────
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import time

# Fix random seed for reproducible weight initialisation across runs
torch.manual_seed(0)

print(f"PyTorch version : {torch.__version__}")
print(f"NumPy version   : {np.__version__}")
print("Environment ready.")

---
<a id="1-environment-setup--dataset"></a>

## 1. Environment Setup & Dataset

### Fashion-MNIST Overview

Fashion-MNIST (Zalando Research, 2017) contains **70,000 grayscale images** across
**10 clothing categories** at 28×28 pixels. It is designed as a direct drop-in
replacement for the original MNIST digit dataset, providing a more challenging
benchmark for image classification algorithms.

| Label | Class | Label | Class |
|---|---|---|---|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

> **Note on image resizing:** Images are resized from 28×28 to **16×16** pixels
> to reduce training time on CPU. This reduces spatial resolution and will yield
> lower accuracy than published 28×28 benchmarks.

In [ ]:
# ── Dataset configuration ────────────────────────────────────────────────────
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Resize from 28×28 to 16×16 — reduces compute cost on CPU
# while retaining enough spatial structure for a 2-layer CNN to learn
IMAGE_SIZE = 16

# Compose the preprocessing pipeline:
#   Resize  — downsample to 16×16 using bilinear interpolation (torchvision default)
#   ToTensor — convert PIL image to float32 tensor, scales pixel values to [0, 1]
composed = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

# Download Fashion-MNIST splits (downloads ~30 MB on first run only)
dataset_train = dsets.FashionMNIST(
    root='.fashion/data', train=True,  transform=composed, download=True
)
dataset_val = dsets.FashionMNIST(
    root='.fashion/data', train=False, transform=composed, download=True
)

print(f"Training samples   : {len(dataset_train):,}")
print(f"Validation samples : {len(dataset_val):,}")
print(f"Image size         : {IMAGE_SIZE}×{IMAGE_SIZE} pixels (resized from 28×28)")
print(f"Tensor shape       : {dataset_train[0][0].shape}  → (C, H, W)")

In [ ]:
# ── Utility: display one Fashion-MNIST sample ────────────────────────────────
def show_data(data_sample, ax=None):
    """
    Display a single Fashion-MNIST image with its class name and numeric label.

    Args:
        data_sample : tuple (image_tensor, label_int) from the dataset
        ax          : optional matplotlib Axes — if None, creates its own figure
    """
    img, label = data_sample
    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))
    # Reshape from (1, H, W) tensor to (H, W) numpy array for imshow
    ax.imshow(img.numpy().reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')
    ax.set_title(f'{CLASS_NAMES[label]}\n(y = {label})', fontsize=10, fontweight='bold')
    ax.axis('off')

#### Task 1 — Display First Three Validation Samples

Display the **first three** validation samples individually, each in its own figure.
Each output shows:
- The resized **16×16** grayscale image
- The **class name** (e.g. Trouser)
- The **numeric label** `y` used during training

In [ ]:
# ── Task 1: display first 3 validation samples — one figure per sample ───────
# Iterating dataset_val directly yields (image_tensor, label) pairs.
# Each sample is rendered in its own figure with plt.show() so outputs
# appear sequentially rather than combined into a single grid.

for n, data_sample in enumerate(dataset_val):
    img, label = data_sample

    fig, ax = plt.subplots(figsize=(3, 3))

    # Display the resized grayscale image
    ax.imshow(img.numpy().reshape(IMAGE_SIZE, IMAGE_SIZE), cmap='gray')

    # Title shows sample index, human-readable class name, and numeric label
    ax.set_title(
        f'Sample index : {n}\n'
        f'Class        : {CLASS_NAMES[label]}\n'
        f'Label (y)    : {label}',
        fontsize=10, fontweight='bold', pad=8
    )

    # X-axis label documents the resize operation for transparency
    ax.set_xlabel(
        f'Resized: {IMAGE_SIZE}×{IMAGE_SIZE} px  (original: 28×28)',
        fontsize=8
    )

    # Remove tick marks — pixel coordinates add no interpretive value here
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # Blue border to visually distinguish individual sample figures
    for spine in ax.spines.values():
        spine.set_edgecolor('steelblue')
        spine.set_linewidth(2)

    plt.tight_layout()
    plt.show()

    # Stop after the third sample (indices 0, 1, 2)
    if n == 2:
        break

#### Extended View — First 9 Validation Samples (Grid)

A 3×3 grid showing the first 9 validation samples together.
This provides a broader view of class diversity and image quality at 16×16 resolution.

In [ ]:
# ── Extended: first 9 validation samples displayed as a labelled 3×3 grid ───
fig, axes = plt.subplots(3, 3, figsize=(8, 8))

for i, ax in enumerate(axes.flatten()):
    show_data(dataset_val[i], ax=ax)

plt.suptitle(
    f'Fashion-MNIST — First 9 Validation Samples\n'
    f'(resized to {IMAGE_SIZE}×{IMAGE_SIZE} from original 28×28)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fashion_mnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fashion_mnist_samples.png")

In [ ]:
# ── DataLoader configuration ─────────────────────────────────────────────────
BATCH_SIZE = 100  # 100 samples per gradient update — balances speed and stability

# shuffle=True for training: ensures each epoch sees samples in a different order,
# which prevents the model from memorising batch-level patterns
train_loader = DataLoader(dataset=dataset_train, batch_size=BATCH_SIZE, shuffle=True)

# shuffle=False for validation: consistent ordering makes results reproducible
test_loader  = DataLoader(dataset=dataset_val,   batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(train_loader)}  ({len(dataset_train):,} samples ÷ {BATCH_SIZE})")
print(f"Test batches  : {len(test_loader)}  ({len(dataset_val):,} samples ÷ {BATCH_SIZE})")

---
<a id="2-model-architectures"></a>

## 2. Model Architectures

Both models share the same two-block convolutional backbone.
`CNN_batch` adds **Batch Normalization** after each convolutional layer and the
final fully-connected layer.

### Spatial dimension flow (16×16 input)

```
Input  (1 × 16 × 16)
  │
  ▼  Conv2d(1→16, kernel=5, padding=2)   → 16 × 16 × 16   [same padding: (16+2×2-5)/1+1=16]
  ▼  [BatchNorm2d(16)]                   → 16 × 16 × 16   (CNN_batch only)
  ▼  ReLU
  ▼  MaxPool2d(kernel=2, stride=2)       → 16 ×  8 ×  8
  │
  ▼  Conv2d(16→32, kernel=5, padding=2)  → 32 ×  8 ×  8   [same padding]
  ▼  [BatchNorm2d(32)]                   → 32 ×  8 ×  8   (CNN_batch only)
  ▼  ReLU
  ▼  MaxPool2d(kernel=2, stride=2)       → 32 ×  4 ×  4
  │
  ▼  Flatten                             →        512      [32 × 4 × 4 = 512]
  ▼  Linear(512 → 10)                   →         10
  ▼  [BatchNorm1d(10)]                   →         10      (CNN_batch only)
  │
  ▼  Output logits (10 classes)
```

### Why these architectural choices?

| Choice | Reason |
|---|---|
| `kernel_size=5, padding=2` | Same padding — output spatial size equals input size at each conv block |
| `out_channels` 1→16→32 | Progressive channel expansion captures increasingly abstract features |
| `MaxPool2d(kernel=2)` | Halves spatial dimensions at each stage — standard 2× downsampling |
| `Linear(32×4×4=512, 10)` | Flattened feature map feeds directly into 10-class classifier |

In [ ]:
# ── Model Definition 1: Standard CNN ─────────────────────────────────────────
class CNN(nn.Module):
    """
    Two-block convolutional network for 10-class Fashion-MNIST classification.

    Architecture: Conv→ReLU→MaxPool (×2) → Flatten → Linear(512→10)
    Input shape : (N, 1, 16, 16)
    Output      : raw logits of shape (N, 10) — passed to CrossEntropyLoss
    """

    def __init__(self, out_1=16, out_2=32, number_of_classes=10):
        super(CNN, self).__init__()

        # ── Convolutional Block 1 ─────────────────────────────────────────────
        # padding=2 with kernel=5 gives same padding: (16+4-5)/1+1 = 16 → stays 16×16
        self.cnn1     = nn.Conv2d(in_channels=1, out_channels=out_1,
                                   kernel_size=5, padding=2)
        # MaxPool halves spatial dims: 16×16 → 8×8
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)

        # ── Convolutional Block 2 ─────────────────────────────────────────────
        self.cnn2     = nn.Conv2d(in_channels=out_1, out_channels=out_2,
                                   kernel_size=5, stride=1, padding=2)
        # MaxPool halves spatial dims: 8×8 → 4×4
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)

        # ── Fully-Connected Classifier ────────────────────────────────────────
        # Flattened feature map: out_2 channels × 4×4 spatial = 32×16 = 512
        self.fc1      = nn.Linear(out_2 * 4 * 4, number_of_classes)

    def forward(self, x):
        # Block 1: Conv → ReLU → Pool
        x = torch.relu(self.cnn1(x))
        x = self.maxpool1(x)
        # Block 2: Conv → ReLU → Pool
        x = torch.relu(self.cnn2(x))
        x = self.maxpool2(x)
        # Flatten (N, 32, 4, 4) → (N, 512) before the linear classifier
        x = x.view(x.size(0), -1)
        return self.fc1(x)


# ── Model Definition 2: CNN with Batch Normalization ─────────────────────────
class CNN_batch(nn.Module):
    """
    Same architecture as CNN with BatchNorm2d after each conv layer
    and BatchNorm1d after the final FC layer.

    Batch Normalization normalises activations across the mini-batch:
      - Reduces internal covariate shift (activations stay in healthy range)
      - Allows higher learning rates without divergence
      - Acts as a mild regulariser (reduces over-reliance on individual neurons)
      - Typically accelerates convergence compared to a plain CNN
    """

    def __init__(self, out_1=16, out_2=32, number_of_classes=10):
        super(CNN_batch, self).__init__()

        # ── Convolutional Block 1 ─────────────────────────────────────────────
        self.cnn1      = nn.Conv2d(in_channels=1, out_channels=out_1,
                                    kernel_size=5, padding=2)
        # BatchNorm2d(C): normalises each of the C=16 channels independently
        # across the batch dimension and both spatial dimensions (H, W)
        self.conv1_bn  = nn.BatchNorm2d(out_1)
        self.maxpool1  = nn.MaxPool2d(kernel_size=2)

        # ── Convolutional Block 2 ─────────────────────────────────────────────
        self.cnn2      = nn.Conv2d(in_channels=out_1, out_channels=out_2,
                                    kernel_size=5, stride=1, padding=2)
        self.conv2_bn  = nn.BatchNorm2d(out_2)
        self.maxpool2  = nn.MaxPool2d(kernel_size=2)

        # ── Fully-Connected Classifier ────────────────────────────────────────
        self.fc1       = nn.Linear(out_2 * 4 * 4, number_of_classes)
        # BatchNorm1d(C): normalises across the batch for each of the C=10 outputs
        self.bn_fc1    = nn.BatchNorm1d(number_of_classes)

    def forward(self, x):
        # Block 1: Conv → BatchNorm → ReLU → Pool
        # Note: BN is applied BEFORE ReLU — the standard convention
        x = torch.relu(self.conv1_bn(self.cnn1(x)))
        x = self.maxpool1(x)
        # Block 2: Conv → BatchNorm → ReLU → Pool
        x = torch.relu(self.conv2_bn(self.cnn2(x)))
        x = self.maxpool2(x)
        # Flatten → FC → BatchNorm (no activation after final layer — CrossEntropy handles it)
        x = x.view(x.size(0), -1)
        return self.bn_fc1(self.fc1(x))


# ── Parameter count comparison ────────────────────────────────────────────────
def count_params(model):
    """Count trainable parameters (weights + biases + BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

torch.manual_seed(0)
model_cnn   = CNN(out_1=16, out_2=32, number_of_classes=10)
model_batch = CNN_batch(out_1=16, out_2=32, number_of_classes=10)

print("Model summary:")
print(f"  CNN       — trainable parameters: {count_params(model_cnn):,}")
print(f"  CNN_batch — trainable parameters: {count_params(model_batch):,}")
print(f"  Difference: +{count_params(model_batch) - count_params(model_cnn)} params")
print("  (BatchNorm adds learnable γ and β per channel — very small overhead)")

---
<a id="3-training-both-models"></a>

## 3. Training Both Models

### Loss function and optimizer

| Component | Choice | Reason |
|---|---|---|
| Loss | `nn.CrossEntropyLoss` | Correct for multi-class classification. Combines LogSoftmax + NLLLoss internally — do not add a softmax layer before this loss |
| Optimizer | `SGD, lr=0.1` | Stochastic Gradient Descent with a relatively high learning rate. Works well for shallow CNNs on simple datasets; Adam would be preferred for deeper networks |
| Epochs | 5 | Sufficient to observe convergence trends on this dataset size at 16×16 resolution |

In [ ]:
# ── Training and evaluation function ─────────────────────────────────────────
def train_and_evaluate(model, train_loader, test_loader, dataset_val,
                       n_epochs=5, lr=0.1):
    """
    Train a model with SGD + CrossEntropyLoss and evaluate on validation set
    after every epoch.

    Args:
        model        : nn.Module instance (CNN or CNN_batch)
        train_loader : DataLoader for training data
        test_loader  : DataLoader for validation data
        dataset_val  : Dataset object (used to compute N_test)
        n_epochs     : number of full passes through the training set
        lr           : SGD learning rate

    Returns:
        cost_list     : list of total training loss per epoch
        accuracy_list : list of validation accuracy (fraction) per epoch
    """
    # CrossEntropyLoss expects raw logits (no softmax) and integer class labels
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    N_test    = len(dataset_val)

    cost_list     = []
    accuracy_list = []
    start_time    = time.time()

    for epoch in range(n_epochs):

        # ── Training phase ────────────────────────────────────────────────────
        # model.train() enables dropout and BatchNorm training behaviour
        # (BN uses batch statistics; dropout randomly zeroes activations)
        model.train()
        cost = 0.0

        for x, y in train_loader:
            optimizer.zero_grad()          # Clear gradients from previous batch
            z    = model(x)               # Forward pass: raw logits
            loss = criterion(z, y)        # CrossEntropy loss
            loss.backward()               # Backpropagation: compute gradients
            optimizer.step()              # Update weights using SGD
            cost += loss.item()           # Accumulate batch loss for epoch total

        # ── Validation phase ──────────────────────────────────────────────────
        # model.eval() disables dropout and switches BN to use running statistics
        # torch.no_grad() disables gradient computation — saves memory and time
        model.eval()
        correct = 0
        with torch.no_grad():
            for x_test, y_test in test_loader:
                z = model(x_test)
                # torch.max returns (values, indices) — indices are predicted classes
                _, yhat = torch.max(z.data, 1)
                correct += (yhat == y_test).sum().item()

        accuracy = correct / N_test
        cost_list.append(cost)
        accuracy_list.append(accuracy)

        elapsed = time.time() - start_time
        print(f"  Epoch {epoch+1}/{n_epochs} | "
              f"Cost: {cost:8.2f} | "
              f"Val Acc: {accuracy*100:.2f}% | "
              f"Elapsed: {elapsed:.1f}s")

    return cost_list, accuracy_list

In [ ]:
# ── Train CNN (standard) ──────────────────────────────────────────────────────
N_EPOCHS = 5
LR       = 0.1

print("=" * 55)
print("Training CNN (standard — no Batch Normalization)")
print("=" * 55)
torch.manual_seed(0)  # Reset seed for fair comparison — identical weight init
model_cnn = CNN(out_1=16, out_2=32, number_of_classes=10)
cost_cnn, acc_cnn = train_and_evaluate(
    model_cnn, train_loader, test_loader, dataset_val,
    n_epochs=N_EPOCHS, lr=LR
)

print()

# ── Train CNN_batch (with Batch Normalization) ────────────────────────────────
print("=" * 55)
print("Training CNN_batch (with Batch Normalization)")
print("=" * 55)
torch.manual_seed(0)  # Same seed — ensures weights start identically
model_batch = CNN_batch(out_1=16, out_2=32, number_of_classes=10)
cost_batch, acc_batch = train_and_evaluate(
    model_batch, train_loader, test_loader, dataset_val,
    n_epochs=N_EPOCHS, lr=LR
)

---
<a id="4-results--comparison"></a>

## 4. Results & Comparison

The two plots below show:
- **Left — Training Cost:** Total CrossEntropy loss summed across all batches per epoch.
  A steeper decline indicates faster learning. BatchNorm typically produces a
  smoother, faster-declining cost curve.
- **Right — Validation Accuracy:** Fraction of correctly classified validation samples
  after each epoch. Higher is better; the gap between the two models (if any)
  quantifies the benefit of Batch Normalization under these hyperparameters.

In [ ]:
# ── Training cost and validation accuracy: CNN vs CNN_batch ──────────────────
epochs = range(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left plot: Training Cost ──────────────────────────────────────────────────
axes[0].plot(epochs, cost_cnn,   'b-o', linewidth=2, markersize=7,
             label='CNN (no BatchNorm)')
axes[0].plot(epochs, cost_batch, 'r-s', linewidth=2, markersize=7,
             label='CNN + BatchNorm')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Total Training Cost (sum of batch losses)', fontsize=11)
axes[0].set_title('Training Cost per Epoch', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# ── Right plot: Validation Accuracy ──────────────────────────────────────────
axes[1].plot(epochs, [a * 100 for a in acc_cnn],
             'b-o', linewidth=2, markersize=7,
             label=f'CNN       (epoch 5: {acc_cnn[-1]*100:.2f}%)')
axes[1].plot(epochs, [a * 100 for a in acc_batch],
             'r-s', linewidth=2, markersize=7,
             label=f'CNN+BN (epoch 5: {acc_batch[-1]*100:.2f}%)')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Validation Accuracy (%)', fontsize=11)
axes[1].set_title('Validation Accuracy per Epoch', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Fashion-MNIST Training: CNN vs CNN + Batch Normalization',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fashion_mnist_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Printed interpretation of results ────────────────────────────────────────
print("Results summary:")
print(f"  CNN       — final validation accuracy : {acc_cnn[-1]*100:.2f}%")
print(f"  CNN_batch — final validation accuracy : {acc_batch[-1]*100:.2f}%")
delta = (acc_batch[-1] - acc_cnn[-1]) * 100
direction = "higher" if delta >= 0 else "lower"
print(f"  CNN_batch is {abs(delta):.2f} percentage points {direction} than CNN")
print("Saved: fashion_mnist_results.png")

---
<a id="5-error-analysis"></a>

## 5. Error Analysis

### Misclassified Samples

Inspecting misclassified samples reveals **which visual patterns the model confuses**.
Common confusions in Fashion-MNIST include:
- Shirt ↔ T-shirt/top (similar silhouette, ambiguous collar)
- Coat ↔ Pullover (both are long upper-body garments)
- Sneaker ↔ Ankle boot (similar shape at 16×16 resolution)

The better-performing model is used for this analysis.

In [ ]:
# ── Misclassification analysis ────────────────────────────────────────────────
def get_misclassified(model, dataset_val, n=9):
    """
    Return up to n misclassified samples from the validation set.

    Returns:
        list of (image_tensor, true_label, predicted_label) tuples
    """
    model.eval()
    misclassified = []
    loader = DataLoader(dataset_val, batch_size=256, shuffle=False)

    with torch.no_grad():
        for x, y in loader:
            z = model(x)
            _, preds = torch.max(z, 1)
            for i in range(len(y)):
                if preds[i].item() != y[i].item():
                    misclassified.append((x[i], y[i].item(), preds[i].item()))
                if len(misclassified) >= n:
                    return misclassified
    return misclassified


# Use the higher-accuracy model for error analysis
best_model = model_batch if acc_batch[-1] >= acc_cnn[-1] else model_cnn
best_name  = 'CNN + BatchNorm' if acc_batch[-1] >= acc_cnn[-1] else 'CNN'
errors     = get_misclassified(best_model, dataset_val, n=9)

print(f"Analysing misclassifications from: {best_name}")
print(f"Misclassified samples retrieved  : {len(errors)}")

if errors:
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for ax, (img, true_lbl, pred_lbl) in zip(axes.flatten(), errors):
        ax.imshow(img.numpy().squeeze(), cmap='gray')
        ax.set_title(
            f'True : {CLASS_NAMES[true_lbl]}\nPred : {CLASS_NAMES[pred_lbl]}',
            fontsize=9, fontweight='bold',
            color='darkred'   # Red title signals an error
        )
        ax.axis('off')
    plt.suptitle(
        f'Misclassified Validation Samples — {best_name}\n'
        f'(True label shown first, predicted label second)',
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('fashion_mnist_errors.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: fashion_mnist_errors.png")
else:
    print("No misclassified samples found — model achieved perfect validation accuracy.")

### Per-Class Accuracy Breakdown

The bar chart below shows validation accuracy for each of the 10 clothing categories.
Classes with lower accuracy (e.g. Shirt, Pullover) are visually similar to other
categories and benefit most from deeper architectures or data augmentation.
This breakdown reveals **where** each model succeeds and fails — information
that the overall accuracy figure alone cannot provide.

In [ ]:
# ── Per-class validation accuracy: CNN vs CNN_batch ──────────────────────────
def per_class_accuracy(model, dataset_val, n_classes=10):
    """
    Compute per-class validation accuracy.

    Returns:
        dict mapping class name (str) → accuracy percentage (float)
    """
    model.eval()
    correct = torch.zeros(n_classes)
    total   = torch.zeros(n_classes)
    loader  = DataLoader(dataset_val, batch_size=256, shuffle=False)

    with torch.no_grad():
        for x, y in loader:
            z = model(x)
            _, preds = torch.max(z, 1)
            for c in range(n_classes):
                mask        = (y == c)
                correct[c] += (preds[mask] == y[mask]).sum().item()
                total[c]   += mask.sum().item()

    return {CLASS_NAMES[c]: (correct[c] / total[c]).item() * 100
            for c in range(n_classes)}


class_acc_cnn   = per_class_accuracy(model_cnn,   dataset_val)
class_acc_batch = per_class_accuracy(model_batch, dataset_val)

# ── Tabular printout ──────────────────────────────────────────────────────────
print(f"{'Class':<18} {'CNN':>9} {'CNN+BN':>9} {'Δ (BN−CNN)':>12}")
print("-" * 52)
for cls in CLASS_NAMES:
    delta = class_acc_batch[cls] - class_acc_cnn[cls]
    sign  = '+' if delta >= 0 else ''
    print(f"{cls:<18} {class_acc_cnn[cls]:>8.1f}%  "
          f"{class_acc_batch[cls]:>8.1f}%  "
          f"{sign}{delta:>8.1f}%")

# ── Grouped bar chart ─────────────────────────────────────────────────────────
x_pos = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))

bars1 = ax.bar(x_pos - width/2,
               [class_acc_cnn[c]   for c in CLASS_NAMES],
               width, label='CNN', color='steelblue', edgecolor='k', alpha=0.85)
bars2 = ax.bar(x_pos + width/2,
               [class_acc_batch[c] for c in CLASS_NAMES],
               width, label='CNN + BatchNorm', color='tomato', edgecolor='k', alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=10)
ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy: CNN vs CNN + Batch Normalization',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='80% reference')

plt.tight_layout()
plt.savefig('fashion_mnist_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fashion_mnist_per_class.png")

---
<a id="6-summary"></a>

## 6. Summary

### Architecture Comparison

| Aspect | CNN | CNN + BatchNorm |
|---|---|---|
| Conv blocks | Conv → ReLU → MaxPool (×2) | Conv → **BN** → ReLU → MaxPool (×2) |
| FC layer | Linear(512 → 10) | Linear(512 → 10) → **BN** |
| Regularisation | None built-in | Mild (BN reduces neuron co-adaptation) |
| Training stability | Standard | Improved (reduced covariate shift) |
| Parameter overhead | Baseline | +learnable γ, β per BN layer (small) |

### Key Takeaways

1. **`nn.CrossEntropyLoss`** is the correct loss for multi-class classification.
   It combines LogSoftmax + NLLLoss — do not add a Softmax layer before it.
2. **Batch Normalization** normalises activations across the mini-batch after each
   convolutional layer, keeping them in a healthy range throughout training.
3. **`model.train()` vs `model.eval()`** is critical when using BatchNorm —
   training mode uses batch statistics; evaluation mode uses running averages.
4. **Per-class accuracy** reveals model weaknesses that overall accuracy hides —
   visually similar classes (Shirt/T-shirt, Coat/Pullover) are consistently harder.

In [ ]:
# ── Final quantitative summary ───────────────────────────────────────────────
print("=" * 55)
print("FINAL RESULTS: Fashion-MNIST CNN Comparison")
print("=" * 55)
print(f"{'Metric':<30} {'CNN':>10} {'CNN+BN':>10}")
print("-" * 55)
print(f"{'Trainable parameters':<30} "
      f"{count_params(model_cnn):>10,} "
      f"{count_params(model_batch):>10,}")
print(f"{'Epoch 1 val accuracy':<30} "
      f"{acc_cnn[0]*100:>9.2f}% "
      f"{acc_batch[0]*100:>9.2f}%")
print(f"{'Epoch 5 val accuracy':<30} "
      f"{acc_cnn[-1]*100:>9.2f}% "
      f"{acc_batch[-1]*100:>9.2f}%")
print(f"{'Epoch 5 training cost':<30} "
      f"{cost_cnn[-1]:>10.2f} "
      f"{cost_batch[-1]:>10.2f}")
print("-" * 55)

winner = 'CNN + BatchNorm' if acc_batch[-1] >= acc_cnn[-1] else 'CNN'
delta  = abs(acc_batch[-1] - acc_cnn[-1]) * 100
print(f"\nHigher accuracy model : {winner}")
print(f"Accuracy difference   : {delta:.2f} percentage points")
print()
print("Figures saved to disk:")
for fname in ['fashion_mnist_samples.png',
              'fashion_mnist_results.png',
              'fashion_mnist_errors.png',
              'fashion_mnist_per_class.png']:
    print(f"  • {fname}")